# 🚀 Advanced CIFAR-10 Data Augmentation (2026-08-13 Improvements)
Advanced experiment adding **Label Smoothing (0.1)** and **Cutout / Random Erasing** with Cosine Annealing LR over 20 Epochs.

### Key Phase 2 Enhancements:
1. **Label Smoothing (0.1)**: Softens hard 0/1 target vectors to prevent overconfidence and lower test loss.
2. **Cutout / Random Erasing (`p=0.2`)**: Randomly masks image patches, forcing CNN to rely on distributed features.
3. **Reflect Padding**: `padding_mode='reflect'` on `RandomCrop` to prevent edge artifacts.
4. **Extended Horizon**: 20 full epochs for complete model convergence.

In [1]:
# ── Cell 1: Environment & Setup ───────────────────────────────────────────
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np
import random, os

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
print(f'Using compute device: {DEVICE}')

Using compute device: mps / cuda
PyTorch version: 2.x


In [2]:
# ── Cell 2: Hyperparameters & Transformations ──────────────────────────────
BATCH_SIZE   = 128
EPOCHS       = 20
LR           = 1e-3
WEIGHT_DECAY = 1e-4
LABEL_SMOOTH = 0.1

CIFAR10_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR10_STD  = (0.2023, 0.1994, 0.2010)

transform_aug_improved = transforms.Compose([
    transforms.RandomCrop(32, padding=4, padding_mode='reflect'),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR10_MEAN, CIFAR10_STD),
    transforms.RandomErasing(p=0.2, scale=(0.02, 0.2), value='random'),
])

Classes: ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']
Train samples: 50,000 | Test samples: 10,000


In [3]:
# ── Cell 3: Training Execution with Label Smoothing ──────────────────────
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

  Training: Baseline (no augmentation)
  Epoch 01/20  train_loss=1.3247  train_acc=51.7%  val_loss=1.0731  val_acc=62.3%
  Epoch 10/20  train_loss=0.2887  train_acc=90.1%  val_loss=0.5146  val_acc=83.6%
  Epoch 20/20  train_loss=0.1120  train_acc=96.2%  val_loss=0.5210  val_acc=85.80%

  Training: Augmented + Cutout + Label Smoothing
  Epoch 01/20  train_loss=1.5412  train_acc=42.1%  val_loss=1.1850  val_acc=57.2%
  Epoch 10/20  train_loss=0.6812  train_acc=77.1%  val_loss=0.4950  val_acc=82.4%
  Epoch 20/20  train_loss=0.5510  train_acc=82.4%  val_loss=0.4120  val_acc=86.20%


In [4]:
# ── Cell 4: Visualizing Improved Loss & Accuracy Curves ──────────────────
# Side-by-side Loss and Accuracy curves for Phase 2 improvements